In [5]:
import joblib
from pathlib import Path
import os
import pandas as pd

In [6]:
IN_KAGGLE = os.path.exists('/kaggle/input')

if IN_KAGGLE:
    # Kaggle غيّر اسم الفولدر ده للأساسي في المسابقة على 
    DATA_DIR = Path('/kaggle/input/house-prices-advanced-regression-techniques')
else:
    DATA_DIR = Path.cwd().parent / "datasets" / "processed"

train_new_data = pd.read_csv(DATA_DIR / 'train_processed.csv')

In [7]:
X_train = train_new_data.drop('SalePrice', axis = 1)

import re
X_train.columns = [re.sub(r'[^A-Za-z0-9_]+', '_', col) for col in X_train.columns]

y_train = train_new_data['SalePrice']

In [8]:


BASE_DIR = Path.cwd().parent
MODELS_DIR = BASE_DIR / "models"

cat_model = joblib.load(MODELS_DIR / "Cat_Boost_Regressor_model.pkl")
gbr_model = joblib.load(MODELS_DIR / "Gradient_Boosting_Regressor_model.pkl")
xgb_model = joblib.load(MODELS_DIR / "XGB_Regressor_model.pkl")
lgbm_model = joblib.load(MODELS_DIR / "LGBM_Regressor_model.pkl")
rf_model = joblib.load(MODELS_DIR / "Random_Forest_Regressor_model.pkl")

print(cat_model)
print(gbr_model)
print(xgb_model)
print(lgbm_model)
print(rf_model)

CatBoostRegressor(bootstrap_type='Bernoulli', depth=8, iterations=1500, l2_leaf_reg=1, learning_rate=0.01, loss_function='RMSE', subsample=0.66, verbose=0)
GradientBoostingRegressor(learning_rate=0.03, max_depth=4, min_samples_leaf=2,
                          min_samples_split=5, n_estimators=200, subsample=0.8)
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=0, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.01, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=3, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=800,
  

In [9]:
from sklearn.model_selection import cross_val_score, KFold
import pandas as pd
import numpy as np

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = {}

for name, model in {
    "CatBoost": cat_model,
    "GradientBoosting": gbr_model,
    "XGBoost": xgb_model,
    "LightGBM": lgbm_model,
    "RandomForest": rf_model
}.items():

    rmse = -cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    ).mean()

    scores[name] = rmse

results = pd.DataFrame.from_dict(
    scores,
    orient="index",
    columns=["RMSE"]
).sort_values("RMSE")

print(results)

                      RMSE
CatBoost          0.258904
XGBoost           0.260238
RandomForest      0.263279
GradientBoosting  0.263894
LightGBM          0.265922
